# Fold-1 Segmentation Engine — Small Dataset Trial
Mirrored/small dataset trial run. See TASK_09_segmentation_engine.md for full spec.
This notebook assumes the repo (or this scaffold) is cloned/uploaded and `data/` follows the contract in the task file.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio pyyaml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust to wherever the repo/scaffold + data/ live in Drive
PROJECT_DIR = '/content/drive/MyDrive/zoning_seg_engine'
%cd {PROJECT_DIR}


In [ ]:
# Sanity check: confirm data contract before burning a training run on
# a malformed dataset
import os
assert os.path.exists('data/images'), 'data/images missing'
assert os.path.exists('data/masks'), 'data/masks missing'
assert os.path.exists('data/splits/train.txt'), 'train split missing'
assert os.path.exists('data/splits/val.txt'), 'val split missing'

import numpy as np
import rasterio
from dataset import ADDON_INDEX
sample_id = open('data/splits/train.txt').readline().strip()
# images/  = addon "<stem>_satellite.png" (RGB); masks/ = "<stem>_mask_index.png"
with rasterio.open(f'data/images/{sample_id}.png') as src:
    print('image shape:', src.read().shape, 'dtype:', src.dtypes)
with rasterio.open(f'data/masks/{sample_id}.png') as src:
    idx = src.read(1)  # single-channel class-index map (multi-class, not 4-band)
    print('mask (index map) shape:', idx.shape, 'dtype:', idx.dtype)
    print('unique class indices present:', np.unique(idx))
    print('Task 09 channels <- addon index:', ADDON_INDEX)
    print('dataset.py expands this single-channel index map into 4 binary channels')

In [ ]:
# For the initial trial: shrink epochs and batch size in config.yaml
# (or override here) to get a fast smoke-test run before a full pass.
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['train']['epochs'] = 10
cfg['train']['batch_size'] = 4
with open('config_trial.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print('Wrote config_trial.yaml for smoke test')


In [ ]:
!python train.py --config config_trial.yaml


## Reading the results
Check per-class IoU independently — do not judge the run on the mean alone.
`parcel_border` may lag the other three; before treating it as a bug, pull
up a few failing validation tiles and check whether the boundary is
actually visible in the imagery (see TASK_09 'known hard case').